In [ ]:
"""================================================================================PART 8: DEPLOYMENT & ETHICS - From Prototype to Value================================================================================Complete Python script covering:1. Model training and serialization (pickle, joblib)2. Flask REST API creation for model deployment3. Fairness auditing and bias detection4. Model card generation and monitoring plan5. Business impact analysisAuthor: Data Science CourseDate: 2024================================================================================"""# ============================================================================# SECTION 1: IMPORTS AND SETUP# ============================================================================import pandas as pdimport numpy as npfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoder, StandardScalerfrom sklearn.metrics import roc_auc_score, confusion_matrix, classification_reportimport pickleimport joblibimport jsonimport warningsimport osfrom datetime import datetimewarnings.filterwarnings('ignore')print("=" * 80)print("PART 8: DEPLOYMENT & ETHICS - From Prototype to Value")print("=" * 80)print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")print("=" * 80)# ============================================================================# SECTION 2: CREATE SYNTHETIC DATASET# ============================================================================# Why: We need realistic data that demonstrates bias patterns# Business context: Credit risk assessment for loan approvalsprint("\n" + "=" * 80)print("STEP 1: DATA GENERATION - Credit Risk Dataset")print("=" * 80)np.random.seed(42)n_samples = 5000print(f"\nGenerating {n_samples} synthetic customer records...")# Generate features with realistic distributionsdata = {    # Demographic features    'age': np.random.randint(18, 70, n_samples),    'gender': np.random.choice(['Male', 'Female'], n_samples, p=[0.6, 0.4]),        # Financial features    'income': np.random.normal(50000, 20000, n_samples).astype(int),    'loan_amount': np.random.normal(15000, 10000, n_samples).astype(int),    'credit_score': np.random.randint(300, 850, n_samples),    'debt_to_income': np.random.uniform(0, 0.5, n_samples),        # Employment features    'employment_years': np.random.randint(0, 40, n_samples),    'late_payments': np.random.poisson(1, n_samples),        # Categorical features    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),    'marital_status': np.random.choice(['Single', 'Married', 'Divorced'], n_samples)}df = pd.DataFrame(data)# Generate target variable (risk: 1 = good credit, 0 = bad credit)# Business logic: Higher risk for:# - Lower credit scores# - Higher debt-to-income ratio# - More late paymentsrisk_score = (    -df['credit_score'] / 1000 +           # Lower score = higher risk    df['debt_to_income'] * 3 +              # High debt = higher risk    df['late_payments'] * 0.5 +             # Late payments = higher risk    np.random.normal(0, 0.3, n_samples)     # Random noise)df['risk'] = (risk_score > np.median(risk_score)).astype(int)print(f"\nвњ… Dataset created successfully!")print(f"   Shape: {df.shape}")print(f"   Features: {list(df.columns)}")print(f"\n   Target distribution:")print(f"   Good credit (1): {df['risk'].sum():,} ({df['risk'].mean()*100:.1f}%)")print(f"   Bad credit (0):  {(len(df)-df['risk'].sum()):,} ({(1-df['risk'].mean())*100:.1f}%)")# Display sampleprint(f"\n   First 5 rows:")print(df.head())# ============================================================================# SECTION 3: PREPROCESSING AND MODEL TRAINING# ============================================================================# Why: Models need numerical, scaled features. We'll train a Random Forest# Business context: Random Forest provides feature importance and handles non-linear relationshipsprint("\n" + "=" * 80)print("STEP 2: PREPROCESSING AND MODEL TRAINING")print("=" * 80)# Separate features and targetX = df.drop('risk', axis=1)y = df['risk']# Identify categorical and numerical columnscategorical_cols = ['gender', 'education', 'marital_status']numerical_cols = ['age', 'income', 'loan_amount', 'credit_score',                   'debt_to_income', 'employment_years', 'late_payments']print(f"\nCategorical features: {categorical_cols}")print(f"Numerical features: {numerical_cols}")# Encode categorical variablesprint("\nEncoding categorical features...")encoders = {}for col in categorical_cols:    le = LabelEncoder()    X[col] = le.fit_transform(X[col])    encoders[col] = le    print(f"   {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")# Scale numerical featuresprint("\nScaling numerical features...")scaler = StandardScaler()X[numerical_cols] = scaler.fit_transform(X[numerical_cols])print(f"   Scaled {len(numerical_cols)} features to zero mean, unit variance")# Split data for validationX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f"\nData split:")print(f"   Training set: {X_train.shape[0]} samples")print(f"   Test set: {X_test.shape[0]} samples")# Train Random Forest modelprint("\nTraining Random Forest model...")model = RandomForestClassifier(    n_estimators=100,    max_depth=10,    min_samples_split=5,    min_samples_leaf=2,    random_state=42,    n_jobs=-1)model.fit(X_train, y_train)# Evaluate modeltrain_score = model.score(X_train, y_train)test_score = model.score(X_test, y_test)y_pred_proba = model.predict_proba(X_test)[:, 1]auc_score = roc_auc_score(y_test, y_pred_proba)print(f"\nвњ… Model trained successfully!")print(f"   Training accuracy: {train_score:.4f}")print(f"   Test accuracy: {test_score:.4f}")print(f"   ROC-AUC: {auc_score:.4f}")# Feature importance analysisfeature_importance = pd.DataFrame({    'feature': X.columns,    'importance': model.feature_importances_}).sort_values('importance', ascending=False)print(f"\nрџ“Љ Top 10 Most Important Features:")print(feature_importance.head(10).to_string(index=False))# ============================================================================# SECTION 4: MODEL SERIALIZATION (SAVING FOR DEPLOYMENT)# ============================================================================# Why: Models must be saved to disk to be loaded by production applications# Business context: The trained model is an asset that needs versioning and storageprint("\n" + "=" * 80)print("STEP 3: MODEL SERIALIZATION - Saving for Deployment")print("=" * 80)# Method 1: Pickle (Python's native serialization)print("\nSaving with pickle...")with open('credit_model.pkl', 'wb') as f:    pickle.dump(model, f)print("   вњ… Saved as 'credit_model.pkl'")# Method 2: Joblib (optimized for scikit-learn models)print("\nSaving with joblib...")joblib.dump(model, 'credit_model.joblib')print("   вњ… Saved as 'credit_model.joblib'")# Save preprocessing objects (CRITICAL for consistency!)print("\nSaving preprocessing objects...")joblib.dump(scaler, 'scaler.joblib')joblib.dump(encoders, 'label_encoders.joblib')print("   вњ… Saved scaler.joblib and label_encoders.joblib")# Verify loading worksprint("\nVerifying model loading...")with open('credit_model.pkl', 'rb') as f:    loaded_model = pickle.load(f)sample = X_test.iloc[[0]]pred_original = model.predict(sample)[0]pred_loaded = loaded_model.predict(sample)[0]if pred_original == pred_loaded:    print("   вњ… Model loads correctly! Ready for deployment.")else:    print("   вќЊ Error: Model loading failed!")# ============================================================================# SECTION 5: FLASK API TEMPLATE GENERATION# ============================================================================# Why: Production models need APIs to serve predictions to applications# Business context: REST API enables integration with web/mobile appsprint("\n" + "=" * 80)print("STEP 4: FLASK API TEMPLATE GENERATION")print("=" * 80)api_code = '''"""CREDIT RISK API - Flask REST EndpointRun with: python app.pyEndpoints:  GET  /          - API information  GET  /health    - Health check  POST /predict   - Get credit risk prediction"""from flask import Flask, request, jsonifyimport joblibimport pandas as pdimport numpy as np# Initialize Flask appapp = Flask(__name__)# Load model and preprocessing objects at startupprint("Loading model and preprocessing objects...")model = joblib.load('credit_model.joblib')scaler = joblib.load('scaler.joblib')encoders = joblib.load('label_encoders.joblib')# Define feature listsnumerical_cols = ['age', 'income', 'loan_amount', 'credit_score',                   'debt_to_income', 'employment_years', 'late_payments']categorical_cols = ['gender', 'education', 'marital_status']@app.route('/')def home():    """API information endpoint"""    return jsonify({        'service': 'Credit Risk Prediction API',        'version': '1.0',        'endpoints': {            '/': 'This information',            '/health': 'Health check',            '/predict': 'POST - Submit customer data for risk prediction'        },        'example_request': {            'age': 45,            'gender': 'Male',            'income': 60000,            'loan_amount': 20000,            'credit_score': 720,            'debt_to_income': 0.3,            'employment_years': 10,            'late_payments': 0,            'education': 'Bachelor',            'marital_status': 'Married'        }    })@app.route('/health', methods=['GET'])def health():    """Health check endpoint"""    return jsonify({        'status': 'healthy',        'model_loaded': True,        'timestamp': str(pd.Timestamp.now())    })@app.route('/predict', methods=['POST'])def predict():    """    Predict credit risk for a customer    Expected JSON with all features    """    try:        # Get JSON data from request        data = request.get_json()                # Validate input        required_fields = numerical_cols + categorical_cols        missing_fields = [f for f in required_fields if f not in data]        if missing_fields:            return jsonify({                'error': f'Missing fields: {missing_fields}',                'message': 'Prediction failed'            }), 400                # Convert to DataFrame        input_df = pd.DataFrame([data])                # Apply categorical encoding (same as training!)        for col in categorical_cols:            if col in input_df.columns:                try:                    input_df[col] = encoders[col].transform(input_df[col])                except ValueError as e:                    return jsonify({                        'error': f'Invalid value for {col}: {data[col]}',                        'message': 'Prediction failed'                    }), 400                # Apply scaling        input_df[numerical_cols] = scaler.transform(input_df[numerical_cols])                # Make prediction        prediction = model.predict(input_df)[0]        probability = model.predict_proba(input_df)[0][1]                # Determine risk level        if probability > 0.7:            risk_level = "Low Risk"            recommendation = "Approve"        elif probability > 0.3:            risk_level = "Medium Risk"            recommendation = "Review Required"        else:            risk_level = "High Risk"            recommendation = "Decline"                # Prepare response        response = {            'prediction': int(prediction),            'probability': float(probability),            'risk_category': 'Good' if prediction == 1 else 'Bad',            'risk_level': risk_level,            'recommendation': recommendation,            'message': 'Success'        }        return jsonify(response)            except Exception as e:        return jsonify({            'error': str(e),            'message': 'Prediction failed'        }), 400if __name__ == '__main__':    app.run(debug=False, host='0.0.0.0', port=5000)'''# Write API file to diskwith open('app.py', 'w') as f:    f.write(api_code)print("\nвњ… Flask API template saved as 'app.py'")print("\nрџ“‹ API Usage Instructions:")print("   1. Install Flask: pip install flask")print("   2. Run API: python app.py")print("   3. Test with curl:")print("      curl -X POST http://localhost:5000/predict \\")print("        -H 'Content-Type: application/json' \\")print("        -d '{\"age\":45,\"gender\":\"Male\",\"income\":60000,\"loan_amount\":20000,\"credit_score\":720,\"debt_to_income\":0.3,\"employment_years\":10,\"late_payments\":0,\"education\":\"Bachelor\",\"marital_status\":\"Married\"}'")# ============================================================================# SECTION 6: FAIRNESS AUDIT - BIAS DETECTION# ============================================================================# Why: Models can discriminate against protected groups# Business context: Regulatory compliance (GDPR, ECOA) and ethical responsibilityprint("\n" + "=" * 80)print("STEP 5: FAIRNESS AUDIT - Bias Detection")print("=" * 80)# Add predictions to dataframe for analysisdf['prediction'] = model.predict(X)df['probability'] = model.predict_proba(X)[:, 1]# Add age groups for demographic analysisdf['age_group'] = pd.cut(    df['age'],     bins=[0, 30, 50, 100],     labels=['Young (<30)', 'Middle (30-50)', 'Senior (>50)'])print("\n" + "-" * 80)print("FAIRNESS METRICS BY DEMOGRAPHIC GROUP")print("-" * 80)# ========== GENDER ANALYSIS ==========print("\nрџ“Љ 1. GENDER ANALYSIS")print("-" * 50)gender_stats = {}for gender in ['Male', 'Female']:    group = df[df['gender'] == gender]    approval_rate = group['prediction'].mean()    avg_prob = group['probability'].mean()    count = len(group)    gender_stats[gender] = {'approval_rate': approval_rate, 'count': count}        print(f"\n{gender}:")    print(f"   Customers: {count}")    print(f"   Approval rate: {approval_rate:.1%}")    print(f"   Avg risk probability: {avg_prob:.3f}")# Calculate Disparate Impact Ratio (DIR)# DIR < 0.8 indicates potential discrimination (80% rule)approval_male = gender_stats['Male']['approval_rate']approval_female = gender_stats['Female']['approval_rate']disparate_impact_gender = min(approval_female, approval_male) / max(approval_female, approval_male)print(f"\nрџ“Љ Disparate Impact Ratio (Female/Male): {disparate_impact_gender:.3f}")print(f"   {'вњ… PASS' if disparate_impact_gender >= 0.8 else 'вљ пёЏ FLAG - Possible discrimination'}")# ========== AGE GROUP ANALYSIS ==========print("\nрџ“Љ 2. AGE GROUP ANALYSIS")print("-" * 50)age_stats = {}for age_group in ['Young (<30)', 'Middle (30-50)', 'Senior (>50)']:    group = df[df['age_group'] == age_group]    approval_rate = group['prediction'].mean()    count = len(group)    age_stats[age_group] = {'approval_rate': approval_rate, 'count': count}        print(f"\n{age_group}:")    print(f"   Customers: {count}")    print(f"   Approval rate: {approval_rate:.1%}")# Compare senior vs middle ageapproval_senior = age_stats['Senior (>50)']['approval_rate']approval_middle = age_stats['Middle (30-50)']['approval_rate']disparate_impact_age = approval_senior / approval_middleprint(f"\nрџ“Љ Disparate Impact Ratio (Senior/Middle): {disparate_impact_age:.3f}")print(f"   {'вњ… PASS' if disparate_impact_age >= 0.8 else 'вљ пёЏ FLAG - Possible age discrimination'}")# ========== ERROR RATE ANALYSIS ==========print("\nрџ“Љ 3. ERROR RATE ANALYSIS (False Positives & Negatives)")print("-" * 50)print("False Positive: Bad loan approved (model said Good, actually Bad)")print("False Negative: Good loan rejected (model said Bad, actually Good)")for gender in ['Male', 'Female']:    group = df[df['gender'] == gender]        # Confusion matrix elements    tp = ((group['prediction'] == 1) & (group['risk'] == 1)).sum()    tn = ((group['prediction'] == 0) & (group['risk'] == 0)).sum()    fp = ((group['prediction'] == 1) & (group['risk'] == 0)).sum()    fn = ((group['prediction'] == 0) & (group['risk'] == 1)).sum()        total_bad = (group['risk'] == 0).sum()    total_good = (group['risk'] == 1).sum()        fp_rate = fp / total_bad if total_bad > 0 else 0    fn_rate = fn / total_good if total_good > 0 else 0        print(f"\n{gender}:")    print(f"   False Positive Rate (bad loans approved): {fp_rate:.3f} ({fp}/{total_bad})")    print(f"   False Negative Rate (good loans rejected): {fn_rate:.3f} ({fn}/{total_good})")# ========== BIAS SUMMARY ==========print("\n" + "=" * 80)print("BIAS SUMMARY AND RECOMMENDATIONS")print("=" * 80)bias_flags = []if disparate_impact_gender < 0.8:    bias_flags.append("Gender disparity detected")if disparate_impact_age < 0.8:    bias_flags.append("Age disparity detected")if bias_flags:    print("\nвљ пёЏ BIAS ALERT: The following issues were detected:")    for flag in bias_flags:        print(f"   вЂў {flag}")    print("\nрџ”§ RECOMMENDED ACTIONS:")    print("   1. Collect more diverse training data")    print("   2. Consider fairness-aware algorithms (e.g., Fairlearn)")    print("   3. Implement human oversight for borderline cases")    print("   4. Add fairness constraints to model optimization")    print("   5. Conduct regular fairness audits")else:    print("\nвњ… FAIRNESS CHECK PASSED: No severe disparate impact detected")    print("   Continue monitoring quarterly for drift")# ============================================================================# SECTION 7: MODEL CARD GENERATION# ============================================================================# Why: Model cards document limitations and intended use# Business context: Transparency for regulators, customers, and internal teamsprint("\n" + "=" * 80)print("STEP 6: MODEL CARD - Documentation")print("=" * 80)model_card = {    "Model Information": {        "Name": "Credit Risk Random Forest v1.0",        "Version": "1.0",        "Date": datetime.now().strftime("%Y-%m-%d"),        "Type": "Binary Classification",        "Algorithm": "Random Forest Classifier"    },    "Intended Use": {        "Purpose": "Predict probability of loan default",        "Scope": "Personal loans under $50,000",        "Users": "Loan officers and credit analysts",        "Decisions": "Decision support tool (not fully automated)"    },    "Training Data": {        "Source": "Synthetic dataset (simulated customer data)",        "Size": f"{n_samples} records",        "Time Period": "Simulated 2020-2023",        "Features": numerical_cols + categorical_cols    },    "Performance Metrics": {        "Accuracy": f"{test_score:.3f}",        "ROC-AUC": f"{auc_score:.3f}",        "Precision": f"{classification_report(y_test, model.predict(X_test), output_dict=True)['1']['precision']:.3f}",        "Recall": f"{classification_report(y_test, model.predict(X_test), output_dict=True)['1']['recall']:.3f}",        "F1-Score": f"{classification_report(y_test, model.predict(X_test), output_dict=True)['1']['f1-score']:.3f}"    },    "Fairness Assessment": {        "Gender Disparate Impact": f"{disparate_impact_gender:.3f}",        "Age Disparate Impact": f"{disparate_impact_age:.3f}",        "Status": "Pass" if len(bias_flags) == 0 else "Review Required"    },    "Known Limitations": [        "May underperform for customers with thin credit files",        "Not validated on economic recession data",        "Uses features that may correlate with protected attributes",        "Synthetic data may not fully represent real-world patterns"    ],    "Recommended Mitigations": [        "Human review for cases with probability between 0.3-0.7",        "Monthly fairness monitoring dashboard",        "Quarterly model retraining with new data",        "Regular bias audits with updated metrics"    ],    "Monitoring Plan": {        "Frequency": "Quarterly",        "Metrics": [            "Disparate Impact Ratio",            "False Positive Rates by group",            "False Negative Rates by group",            "Model accuracy drift"        ],        "Alert Thresholds": {            "Disparate Impact": "< 0.8 for 2 consecutive quarters",            "Accuracy Drop": "> 5% decline",            "Data Drift": "PS > 0.1"        }    }}# Save model card to JSONwith open('model_card.json', 'w') as f:    json.dump(model_card, f, indent=2)print("\nвњ… Model card saved as 'model_card.json'")print("\nрџ“‹ Model Card Summary:")for section, content in model_card.items():    print(f"\n{section}:")    if isinstance(content, dict):        for key, value in content.items():            print(f"   {key}: {value}")    elif isinstance(content, list):        for item in content:            print(f"   вЂў {item}")    else:        print(f"   {content}")# ============================================================================# SECTION 8: BUSINESS IMPACT ANALYSIS# ============================================================================# Why: Models need to demonstrate business value# Business context: ROI calculation for stakeholdersprint("\n" + "=" * 80)print("STEP 7: BUSINESS IMPACT ANALYSIS")print("=" * 80)# Simulate business costs and benefitscost_per_call = 2.50      # Cost to contact a customerdeposit_value = 50.00     # Profit from successful deposittotal_customers = len(X_test)# Get predictions on test sety_pred = model.predict(X_test)y_proba = model.predict_proba(X_test)[:, 1]print("\nрџ’° CAMPAIGN COST-BENEFIT ANALYSIS")print("-" * 60)# Strategy 1: Call everyone (baseline)calls_all = total_customerscost_all = calls_all * cost_per_callsuccesses_all = y_test.sum()profit_all = successes_all * deposit_value - cost_allprint(f"\nрџ“ћ Strategy 1: Call EVERYONE")print(f"   Calls: {calls_all}")print(f"   Cost: ${cost_all:,.2f}")print(f"   Expected successes: {successes_all}")print(f"   Profit: ${profit_all:,.2f}")# Strategy 2: Target top 30% by model probabilitythreshold_70 = np.percentile(y_proba, 70)targeted_customers = y_proba >= threshold_70calls_targeted = targeted_customers.sum()cost_targeted = calls_targeted * cost_per_callsuccesses_targeted = y_test[targeted_customers].sum()profit_targeted = successes_targeted * deposit_value - cost_targetedprint(f"\nрџЋЇ Strategy 2: Model-targeted (Top 30%)")print(f"   Calls: {calls_targeted}")print(f"   Cost: ${cost_targeted:,.2f}")print(f"   Expected successes: {successes_targeted}")print(f"   Profit: ${profit_targeted:,.2f}")# Strategy 3: Target top 50% by model probabilitythreshold_50 = np.percentile(y_proba, 50)targeted_customers_50 = y_proba >= threshold_50calls_targeted_50 = targeted_customers_50.sum()cost_targeted_50 = calls_targeted_50 * cost_per_callsuccesses_targeted_50 = y_test[targeted_customers_50].sum()profit_targeted_50 = successes_targeted_50 * deposit_value - cost_targeted_50print(f"\nрџЋЇ Strategy 3: Model-targeted (Top 50%)")print(f"   Calls: {calls_targeted_50}")print(f"   Cost: ${cost_targeted_50:,.2f}")print(f"   Expected successes: {successes_targeted_50}")print(f"   Profit: ${profit_targeted_50:,.2f}")# Calculate ROI and improvementprint("\nрџ“€ BUSINESS IMPACT SUMMARY")print("-" * 60)best_strategy = "Top 30% Targeting"best_profit = profit_targetedimprovement = profit_targeted - profit_allroi_improvement = ((profit_targeted / cost_targeted) / (profit_all / cost_all) - 1) * 100print(f"\nвњ… Best Strategy: {best_strategy}")print(f"   Profit improvement over baseline: ${improvement:,.2f}")print(f"   ROI improvement: {roi_improvement:.1f}%")print(f"   Calls reduced: {calls_all - calls_targeted} ({(calls_all - calls_targeted)/calls_all*100:.1f}%)")print("\nрџ’Ў KEY INSIGHT:")print("   The model doesn't just predict better - it saves money!")print("   By targeting only the most promising customers, we achieve")print("   higher profit with fewer resources.")# ============================================================================# SECTION 9: MONITORING PLAN# ============================================================================# Why: Models degrade over time; need ongoing monitoring# Business context: Proactive risk managementprint("\n" + "=" * 80)print("STEP 8: MONITORING PLAN - Ongoing Oversight")print("=" * 80)monitoring_plan = """CREDIT RISK MODEL MONITORING PLAN v1.0========================================1. QUARTERLY METRICS TO TRACK   ---------------------------   a) Performance Metrics:      - ROC-AUC (target > 0.85)      - Accuracy (target > 0.80)      - F1 Score (target > 0.80)      b) Fairness Metrics:      - Disparate Impact Ratio by gender (target > 0.8)      - Disparate Impact Ratio by age (target > 0.8)      - False Positive Rate parity (difference < 0.05)      - False Negative Rate parity (difference < 0.05)      c) Business Metrics:      - Profit per campaign      - Cost per acquisition      - Conversion rate by segment2. DATA COLLECTION   ----------------   [вњ“] Log all predictions with timestamp   [вњ“] Store input features for drift detection   [вњ“] Track actual outcomes (loan performance)   [вњ“] Collect demographic data with consent3. ALERT THRESHOLDS   -----------------   [GREEN] Green (Normal): All metrics within targets   [YELLOW] Yellow (Warning): Any metric exceeds threshold by 10%   [RED] Red (Critical): DIR < 0.7 or accuracy drop > 10%4. ACTION PROTOCOLS   -----------------   Yellow Alert:   - Investigate root cause   - Review recent data changes   - Increase monitoring frequency      Red Alert:   - Pause automated decisions   - Convene fairness review committee   - Retrain model with updated data   - Conduct full bias audit5. GOVERNANCE STRUCTURE   ---------------------   * Data Science Lead: Model performance monitoring   * Compliance Officer: Fairness and regulatory review   * Business Stakeholder: Business impact assessment   * External Auditor: Annual independent review"""print(monitoring_plan)# Save monitoring plan to file with UTF-8 encodingwith open('monitoring_plan.txt', 'w', encoding='utf-8') as f:    f.write(monitoring_plan)print("\nвњ… Monitoring plan saved as 'monitoring_plan.txt'")# ============================================================================# SECTION 10: FINAL SUMMARY AND DELIVERABLES# ============================================================================print("\n" + "=" * 80)print("EXERCISE COMPLETE - DELIVERABLES SUMMARY")print("=" * 80)deliverables = [    ("credit_model.pkl", "Pickle-serialized model file"),    ("credit_model.joblib", "Joblib-serialized model file"),    ("scaler.joblib", "Feature scaler for preprocessing"),    ("label_encoders.joblib", "Categorical encoders"),    ("app.py", "Flask REST API for model deployment"),    ("model_card.json", "Model documentation and limitations"),    ("monitoring_plan.txt", "Quarterly fairness monitoring plan")]print("\nрџ“Ѓ Generated Files:")for filename, description in deliverables:    exists = "вњ…" if os.path.exists(filename) else "вќЊ"    print(f"   {exists} {filename:30s} - {description}")print("\n" + "=" * 80)print("KEY TAKEAWAYS")print("=" * 80)print("""1. DEPLOYMENT CREATES VALUE: A model in a notebook has zero value.   Real value comes from integrating models into business processes.2. MODELS CAN PERPETUATE HARM: Historical bias in data leads to   discriminatory predictions. Always audit for fairness.3. FAIRNESS IS MEASURABLE: Use metrics like Disparate Impact Ratio,   false positive rates, and subgroup analysis to detect bias.4. TRANSPARENCY BUILDS TRUST: Document limitations in model cards   and be able to explain decisions to stakeholders.5. MONITORING IS CONTINUOUS: Models degrade over time. Implement   ongoing monitoring for performance and fairness drift.6. ETHICS IS GOOD BUSINESS: Fair, transparent models build customer   trust and protect against regulatory and reputational risk.""")print("\n" + "=" * 80)print(f"вњ… EXERCISE COMPLETED SUCCESSFULLY at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")print("=" * 80)